# CME Futures: Risk Overlays

For each return horizon, this notebook selects the highest validation Sharpe from the immutable
union of signal and allocation results, then applies every position-level risk rule declared in
the case-study configuration. Stop-loss, trailing-stop, and time-exit parameters are fixed before
the validation backtest. They are not calibrated from the same validation price path they assess.

Risk rules execute inside the existing futures engine after product-keyed target decisions cross
the typed boundary. Every declared rule must finish, and the resulting per-label candidate sets
remain eligible for final validation selection.

In [1]:
"""Run the declared CME futures risk-overlay population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    pre_overlay_candidate_set,
    product_universe_table,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import get_position_risk_controls

## Fixed per-label inputs and risk rules

No candidate cap or runtime-dependent skip is allowed. The configured list is the population.

In [2]:
study = open_study(execution_tier="canonical")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [3]:
risk_controls = get_position_risk_controls("cme_futures")
if not risk_controls:
    raise ValueError("the configured position-risk population is empty")

request_rows = []
for label in ALL_LABELS:
    selected = pre_overlay_candidate_set(study, label=label).best_validation_sharpe()
    strategy = selected.spec()["strategy"]
    prediction_hash = selected.registry_record()["prediction_hash"]
    for control in risk_controls:
        rule = {key: value for key, value in control.items() if key != "name"}
        request_rows.append(
            {
                "request_name": f"{selected.hash}-risk-{control['name']}",
                "prediction_hash": prediction_hash,
                "label": label,
                "signal": strategy["signal"],
                "allocation": strategy.get("allocation"),
                "risk": {"position_rules": [rule]},
                "costs": None,
                "chapter": "ch19",
            }
        )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "risk")

request_name,prediction_hash,label,risk
str,str,str,object
"""9bc90383acb6-risk-stop_loss_3p…","""28e50f1ddb6d""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.03}]}"
"""9bc90383acb6-risk-stop_loss_5p…","""28e50f1ddb6d""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.05}]}"
"""9bc90383acb6-risk-stop_loss_10…","""28e50f1ddb6d""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.1}]}"
"""9bc90383acb6-risk-stop_loss_15…","""28e50f1ddb6d""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.15}]}"
"""9bc90383acb6-risk-trailing_1pc…","""28e50f1ddb6d""","""fwd_ret_5d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.01}]}"
…,…,…,…
"""48200fc5abeb-risk-trailing_15p…","""206874caf483""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.15}]}"
"""48200fc5abeb-risk-trailing_20p…","""206874caf483""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.2}]}"
"""48200fc5abeb-risk-time_exit_10""","""206874caf483""","""fwd_ret_21d""","{'position_rules': [{'type': 'time_exit', 'bars': 10}]}"


## Execute and freeze risk candidates

Each request carries the fitted prediction checkpoint, product decisions, fold-transition policy,
contract and roll inputs, and one risk rule. Missing members fail before the candidate set exists.

In [4]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name="cme_futures-risk-validation-v1",
)
candidate_sets = create_label_candidate_sets(
    study,
    execution,
    stage="risk",
)

In [5]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete
str,str,str,str,str,bool
"""48200fc5abeb-risk-stop_loss_10…","""fwd_ret_21d""","""206874caf483""","""aaf2f9db3c87""","""d4f536faf6fa""",true
"""48200fc5abeb-risk-stop_loss_15…","""fwd_ret_21d""","""206874caf483""","""650ab4213a8f""","""042ad3766e9e""",true
"""48200fc5abeb-risk-stop_loss_3p…","""fwd_ret_21d""","""206874caf483""","""c1230226db10""","""8015bbfbcfba""",true
"""48200fc5abeb-risk-stop_loss_5p…","""fwd_ret_21d""","""206874caf483""","""8af2a08ae0fc""","""ce2a1e6995f1""",true
"""48200fc5abeb-risk-time_exit_10""","""fwd_ret_21d""","""206874caf483""","""3d7a5d1ea5c8""","""a324a06a6aca""",true
…,…,…,…,…,…
"""9bc90383acb6-risk-trailing_1pc…","""fwd_ret_5d""","""28e50f1ddb6d""","""e60c88525107""","""d428fceeaeb9""",true
"""9bc90383acb6-risk-trailing_20p…","""fwd_ret_5d""","""28e50f1ddb6d""","""e1e241f43dce""","""6579787454f3""",true
"""9bc90383acb6-risk-trailing_2pc…","""fwd_ret_5d""","""28e50f1ddb6d""","""6284cd206cb0""","""30fd2881aaa9""",true


Final selection in `17_strategy_analysis` uses the union of signal, allocation, and risk-overlay
results. Cost-sensitivity rows are excluded.